# 05 — Hessam: Integration, Final JSON & Report Assets

**Google Colab notebook.** Runtime → *Change runtime type* → **GPU (T4 / L4 / A100)** before running.

Merges every member's outputs into the final per-invoice JSON, builds the comparison/evaluation charts for the report and deck, and packages the Streamlit demo. **Run this last.**

| | |
|---|---|
| **Inputs** | All members' outputs from Drive `inputs/upstream/` + `outputs/` |
| **Outputs** | `sample_invoice_outputs/*.json`, `final_pipeline_report.md`, report charts |
| **Expected runtime** | ~10–20 min (no training — GPU optional) |
| **Compute profile** | `colab_gpu` (generous — full data, pinned in the profile cell) |

### How results get back to the team
Everything is written to Google Drive by `colab_bootstrap.publish()`, into **both**:
- `outputs/hessam/<kind>/` — the *latest* copy
- `runs/hessam/<UTC-timestamp>/<kind>/` — an immutable archive, so re-running never
  silently destroys an earlier result

Tell the integrator (Hessam) when you're done; he copies from `outputs/` into the repo.

> **Before you run:** `MyDrive/DL2_InvoiceAI/` must already contain `code/` (the repo's `src/`,
> `scripts/`, and `colab_bootstrap.py`) and `inputs/`. If it doesn't, the bootstrap cell fails
> fast with a message telling you exactly what's missing.

### Run order
This notebook consumes what the other four publish. Before running, confirm Drive has:

```
inputs/upstream/diana/stamp_signature_predictions.csv
inputs/upstream/jordan/region_predictions.csv
inputs/upstream/damir/{ocr_outputs,parameter_presence_results,terms_extraction_results}.csv
```

Missing pieces don't crash the notebook — it degrades gracefully and records exactly what was
absent, so you can integrate partial results and re-run later.

In [ ]:
# --- GPU check: stop here if this says "no GPU" ---------------------------------
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or
      "!! NO GPU. Runtime > Change runtime type > Hardware accelerator = GPU, then re-run.")
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
# --- Mount Drive + load the shared bootstrap -----------------------------------
DRIVE_ROOT = "/content/drive/MyDrive/DL2_InvoiceAI"   # <-- change if your folder differs

import sys, os, shutil, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

_bs = Path(DRIVE_ROOT) / "code" / "colab_bootstrap.py"
assert _bs.exists(), (
    f"Missing {_bs}.\nUpload the repo's colab/colab_bootstrap.py into "
    f"{DRIVE_ROOT}/code/ and re-run this cell."
)
sys.path.insert(0, str(_bs.parent))
import colab_bootstrap as CB

root  = CB.mount_drive(DRIVE_ROOT)
paths = CB.setup_paths(root)
CB.install_deps("pandas", "matplotlib")
print("Drive root:", root)

In [ ]:
# --- Pin the generous Colab budget --------------------------------------------
os.environ["IIP_COMPUTE_PROFILE"] = "colab_gpu"
from src.compute_profile import get_profile

P = get_profile()
print(json.dumps(P, indent=2, default=str))

RUN_TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())   # one archive folder for this run
T0 = time.time()

In [ ]:
# --- Collect whatever upstream members have published ------------------------
import pandas as pd, numpy as np

UP = paths.inputs / "upstream"
WANT = {
    "diana_preds":  UP/"diana"/"stamp_signature_predictions.csv",
    "jordan_preds": UP/"jordan"/"region_predictions.csv",
    "damir_ocr":    UP/"damir"/"ocr_outputs.csv",
    "damir_params": UP/"damir"/"parameter_presence_results.csv",
    "damir_terms":  UP/"damir"/"terms_extraction_results.csv",
}
got, missing = {}, []
for k, p in WANT.items():
    if p.exists():
        got[k] = pd.read_csv(p)
        print(f"  OK      {k:14s} {len(got[k]):6d} rows")
    else:
        missing.append(k)
        print(f"  MISSING {k:14s} ({p})")

man = pd.read_csv(paths.inputs/"invoice_manifest.csv")
print(f"\nmanifest: {len(man)} invoices | missing upstream: {missing or 'none'}")

In [ ]:
# --- Build the final per-invoice JSON ----------------------------------------
try:
    from src.final_json_builder import build_final_json      # preferred: shared module
    HAVE_BUILDER = True
except Exception as e:
    HAVE_BUILDER = False
    print("final_json_builder unavailable, using inline fallback:", e)

OUTJ = Path("/content/out/final_json/sample_invoice_outputs")
OUTJ.mkdir(parents=True, exist_ok=True)

def rows_for(df, doc, col="document_id"):
    return [] if df is None or col not in df else df[df[col] == doc].to_dict("records")

SAMPLE = man.head(50)          # a representative sample for the deliverable
for r in SAMPLE.itertuples():
    doc = r.document_id
    stamps = [x for x in rows_for(got.get("diana_preds"), doc) if x.get("label") == "stamp"]
    sigs   = [x for x in rows_for(got.get("diana_preds"), doc) if x.get("label") == "signature"]
    regions = rows_for(got.get("jordan_preds"), doc)
    ocr = rows_for(got.get("damir_ocr"), doc)
    params = rows_for(got.get("damir_params"), doc)

    payload = {
        "document_id": doc,
        "image_path": r.image_path,
        "dimensions": {"width": int(r.width), "height": int(r.height)},
        "stamp": {"detected": bool(stamps), "count": len(stamps), "boxes": stamps},
        "signature": {"detected": bool(sigs), "count": len(sigs), "boxes": sigs},
        "regions": regions,
        "ocr": {"text": (ocr[0].get("ocr_text") if ocr else None),
                "mean_confidence": (ocr[0].get("mean_confidence") if ocr else None)},
        "business_parameters": (params[0] if params else {}),
        "pistac_ready": bool(stamps) and bool(sigs),
        "_provenance": {"upstream_present": sorted(got), "upstream_missing": missing},
    }
    if HAVE_BUILDER:
        try:
            payload = build_final_json(payload)
        except Exception:
            pass
    (OUTJ/f"{doc}.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("wrote", len(list(OUTJ.glob('*.json'))), "JSON files")
print(json.dumps(json.loads(next(OUTJ.glob('*.json')).read_text()), indent=2)[:700])

In [ ]:
# --- Gather every members' metrics for the comparison charts -----------------
metrics = {}
for mem in ["rolando", "diana", "jordan", "damir"]:
    d = paths.outputs(mem)/"metrics"
    for f in sorted(d.glob("*.json")):
        try:
            metrics[f"{mem}/{f.name}"] = json.loads(f.read_text())
        except Exception as e:
            print("  unreadable:", f, e)
print("metrics files found:")
for k in metrics:
    print("  ", k)

# The _run block is what makes cross-run comparison possible.
runs = []
for k, v in metrics.items():
    rb = v.get("_run")
    if rb:
        runs.append({"source": k, **{kk: rb.get(kk) for kk in
                    ["member", "profile", "device", "epochs", "imgsz", "batch",
                     "wall_clock_sec", "model", "timestamp_utc"]}})
runs_df = pd.DataFrame(runs)
print("\n", runs_df if not runs_df.empty else "no _run blocks yet")

In [ ]:
# --- Report charts (also written to outputs/_shared/) ------------------------
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    from src.reporting_charts import save_figure, per_class_metrics_bar, profile_comparison
    SHARED = True
except Exception:
    SHARED = False

FIG = Path("/content/out/figures"); FIG.mkdir(parents=True, exist_ok=True)

# 1) per-class detection quality across members
per_class = {}
for k, v in metrics.items():
    if "stamp_signature" in k:
        for cls in ("stamp", "signature"):
            if cls in v:
                per_class[cls] = v[cls]
    if "region_iou" in k and "per_class" in v:
        per_class.update(v["per_class"])

if per_class:
    labels = list(per_class)
    fig, ax = plt.subplots(figsize=(max(7, 1.3*len(labels)), 4.5))
    x = np.arange(len(labels)); w = .27
    for i, (mkey, lbl) in enumerate([("precision", "Precision"), ("recall", "Recall"),
                                     ("mean_iou", "Mean IoU")]):
        ax.bar(x + (i-1)*w, [per_class[l].get(mkey, 0) or 0 for l in labels], w, label=lbl)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1); ax.set_ylabel("score"); ax.legend(frameon=False)
    ax.set_title("Detection quality by class (real held-out data)")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(FIG/"model_comparison_per_class.png", dpi=200); plt.close(fig)
    print("wrote model_comparison_per_class.png")

# 2) runtime / budget comparison across runs
if not runs_df.empty and runs_df.wall_clock_sec.notna().any():
    fig, ax = plt.subplots(figsize=(7, 4))
    d = runs_df.dropna(subset=["wall_clock_sec"])
    ax.barh(d["source"], d["wall_clock_sec"]/60)
    ax.set_xlabel("wall clock (minutes)"); ax.set_title("Run cost by stage / profile")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(FIG/"model_comparison_runtime.png", dpi=200); plt.close(fig)
    print("wrote model_comparison_runtime.png")

for f in FIG.glob("*.png"):
    shutil.copyfile(f, paths.shared("model_comparison")/f.name)
    shutil.copyfile(f, paths.shared("report_assets")/f.name)
print("charts ->", paths.shared("model_comparison"))

In [ ]:
# --- Provenance: the _run block makes cross-run model comparison possible ------
def run_block(**kw):
    """Stamp every metrics JSON with how it was produced, so local-CPU and Colab-GPU
    results can be charted against each other later."""
    b = {
        "profile": P.get("profile_name", "colab_gpu"),
        "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
        "epochs": P.get("epochs"), "imgsz": P.get("imgsz"), "batch": P.get("batch"),
        "wall_clock_sec": round(time.time() - T0, 1),
        "timestamp_utc": RUN_TS, "member": "hessam",
    }
    b.update(kw)
    return b


In [ ]:
# --- Final pipeline report ----------------------------------------------------
ready = sum(1 for f in OUTJ.glob("*.json") if json.loads(f.read_text()).get("pistac_ready"))
REP = Path("/content/out/final_pipeline_report.md")
REP.write_text(f'''# Final Pipeline Report

Generated {RUN_TS} (Colab, profile `colab_gpu`).

## Coverage
- Invoices in manifest: **{len(man)}**
- Final JSON produced for: **{len(list(OUTJ.glob("*.json")))}** (sample)
- Both stamp AND signature detected ("Pistac.io ready"): **{ready}**

## Upstream stages integrated
{chr(10).join(f"- OK {k}" for k in sorted(got)) or "- (none)"}

## Missing / not yet run
{chr(10).join(f"- MISSING {k}" for k in missing) or "- none"}

## Runs compared
{runs_df.to_markdown(index=False) if not runs_df.empty else "_no _run blocks found yet_"}

## Caveats that must appear in the group report
- Diana's detector is trained on SignverOD/StaVer, **not** invoices — a real domain gap.
- Jordan's regions come from the OCR Dataset (**receipts**, ~460px wide) while the invoice
  corpus is full-page 1654x2339. Transfer is imperfect by construction.
- Damir's secondary invoice OCR score has a small denominator (only ~197 of 750 manifest
  images carry ground truth; annotations exist for batch_1 only).
- `batch_3/` duplicates batches 1 and 2; those copies are excluded from the manifest to
  prevent train/test leakage. True unique invoice count is 5,201, not 8,181.
''', encoding="utf-8")
print(REP.read_text()[:1000])

In [ ]:
# --- Publish to Drive (latest + immutable archive) -----------------------------
# 
to_publish = [
    ("logs", REP),
    ("predictions", Path("/content/out/final_json")),
]
for kind, src in to_publish:
    if src is None:
        continue
    p = Path(src)
    if not p.exists():
        print(f"  skip (not produced): {p}")
        continue
    CB.publish("hessam", p, kind, paths=paths, run_timestamp=RUN_TS)

print("\nLatest ->", paths.outputs("hessam"))
print("Archive ->", paths.run_dir("hessam", timestamp=RUN_TS))

In [ ]:
# --- Streamlit demo package ---------------------------------------------------
APP = Path("/content/out/app"); APP.mkdir(parents=True, exist_ok=True)
src_app = paths.code/"app"/"streamlit_app.py"
if src_app.exists():
    shutil.copyfile(src_app, APP/"streamlit_app.py")
    print("copied streamlit_app.py from Drive code/")
else:
    print("NOTE: app/streamlit_app.py not in Drive code/ - copy it there, or run the app "
          "from the local repo against these JSON outputs.")

(APP/"RUN_LOCALLY.md").write_text(f'''# Running the demo locally

1. Copy `outputs/hessam/predictions/final_json/sample_invoice_outputs/*.json` from Drive into
   the repo at `outputs/final_json/sample_invoice_outputs/`.
2. Copy each member's CSVs from `outputs/<member>/predictions/` into `outputs/predictions/`.
3. From the repo root:  `streamlit run app/streamlit_app.py`

Generated {RUN_TS}.
''', encoding="utf-8")
CB.publish("hessam", APP, "logs", paths=paths, run_timestamp=RUN_TS)
print("done")

## Report log — fill this in before you finish

Copy your answers into `presentation/member_reports/hessam_report_log.md` in the repo (or paste
them to the integrator). This is the raw material for the group report and slide deck, so be
specific and **honest about what didn't work**.

1. Which stages were integrated vs missing, and how you handled the gaps.
2. The end-to-end story: how many invoices came out Pistac.io-ready, and what blocked the rest.
3. Which caveat you judge most important for the audience to understand.
4. What you'd do next with more compute or better ground truth.

Also note anything the next stage needs from you, and which figure you'd put on a slide.